In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [3]:
df = pd.read_csv("../data/view/fourth_down_with_features.csv")
print(df.shape)
print(df.columns.tolist())

(35871, 45)
['game_id', 'season', 'week', 'game_date', 'home_team', 'away_team', 'posteam', 'defteam', 'qtr', 'game_seconds_remaining', 'yardline_100', 'ydstogo', 'score_differential', 'roof', 'temp', 'wind', 'shotgun', 'no_huddle', 'qb_dropback', 'qb_scramble', 'epa', 'wp', 'wpa', 'decision', 'go_converted', 'go_failed', 'fg_made', 'kick_distance', 'punt_attempt', 'field_goal_attempt', 'yards_gained', 'touchback', 'drive', 'fixed_drive', 'order_sequence', 'play_id', 'ydstogo_bin', 'yardline_bin', 'go_conv_rate', 'go_attempts', 'go_stop_rate', 'go_faced', 'epa_per_drive_roll6', 'success_rate_roll6', 'points_per_drive_roll6']


In [4]:
punts = df[df["punt_attempt"] == 1].copy()
print(punts.shape)
print(punts[["yardline_100", "yards_gained", "touchback", "posteam", "season", "week"]].head(10))

(19640, 45)
    yardline_100  yards_gained  touchback posteam  season  week
0           39.0           0.0        0.0     JAX    2017     5
2           56.0           0.0        0.0     CAR    2017     8
3           53.0           0.0        0.0     CAR    2017     9
4           46.0           0.0        0.0     CAR    2017     9
5           46.0           0.0        0.0     PHI    2017    13
6           42.0           0.0        0.0     DAL    2017    13
7           46.0           0.0        0.0      KC    2017    14
8           73.0           0.0        0.0     SEA    2017    15
9           74.0           0.0        0.0     SEA    2017    15
10          74.0           0.0        0.0     SEA    2017    15


In [5]:
print(punts[["yardline_100", "kick_distance", "touchback", "posteam", "season", "week"]].head(10))
print(punts["kick_distance"].describe())

    yardline_100  kick_distance  touchback posteam  season  week
0           39.0           35.0        0.0     JAX    2017     5
2           56.0           48.0        0.0     CAR    2017     8
3           53.0           50.0        0.0     CAR    2017     9
4           46.0           38.0        0.0     CAR    2017     9
5           46.0           43.0        0.0     PHI    2017    13
6           42.0           41.0        0.0     DAL    2017    13
7           46.0           13.0        0.0      KC    2017    14
8           73.0           46.0        0.0     SEA    2017    15
9           74.0           41.0        0.0     SEA    2017    15
10          74.0           61.0        0.0     SEA    2017    15
count    19640.000000
mean        45.919959
std          9.785770
min         -1.000000
25%         40.000000
50%         46.000000
75%         52.000000
max         84.000000
Name: kick_distance, dtype: float64


In [6]:
punts["opponent_start"] = 100 - punts["yardline_100"] + punts["kick_distance"]
punts.loc[punts["touchback"] == 1, "opponent_start"] = 80

print(punts["opponent_start"].describe())

count    19640.000000
mean        78.688646
std         12.248914
min          0.000000
25%         72.000000
50%         80.000000
75%         88.000000
max        105.000000
Name: opponent_start, dtype: float64


In [7]:
punts = punts[(punts["kick_distance"] > 0) & (punts["opponent_start"] <= 100)].copy()

print(punts.shape)
print(punts["opponent_start"].describe())

(19530, 46)
count    19530.000000
mean        78.934921
std         11.726626
min         15.000000
25%         72.000000
50%         80.000000
75%         88.000000
max        100.000000
Name: opponent_start, dtype: float64


In [8]:
game_punt_stats = punts.groupby(["posteam", "season", "week"]).agg(
    avg_punt_distance=("kick_distance", "mean"),
    inside_twenty_rate=("opponent_start", lambda x: (x > 80).mean())
).reset_index()

print(game_punt_stats.head(10))

  posteam  season  week  avg_punt_distance  inside_twenty_rate
0     ARI    2016     1          36.000000            0.600000
1     ARI    2016     2          46.500000            0.500000
2     ARI    2016     3          35.333333            0.333333
3     ARI    2016     4          42.250000            0.500000
4     ARI    2016     5          42.444444            0.666667
5     ARI    2016     6          40.000000            0.400000
6     ARI    2016     7          38.166667            0.500000
7     ARI    2016     8          43.166667            0.166667
8     ARI    2016    10          44.250000            0.250000
9     ARI    2016    11          44.800000            0.600000


### Rolling Average Compute

In [9]:
game_punt_stats = game_punt_stats.sort_values(["posteam","season","week"]).copy()

game_punt_stats["punt_distance_roll6"] = (game_punt_stats.groupby("posteam")["avg_punt_distance"].transform(lambda x: x.shift(1).rolling(6, min_periods = 1).mean()))

game_punt_stats["inside_twenty_rate_roll6"] = (game_punt_stats.groupby("posteam")["inside_twenty_rate"].transform(lambda x: x.shift(1).rolling(6, min_periods=1).mean()))

print(game_punt_stats.head(10))

  posteam  season  week  avg_punt_distance  inside_twenty_rate  \
0     ARI    2016     1          36.000000            0.600000   
1     ARI    2016     2          46.500000            0.500000   
2     ARI    2016     3          35.333333            0.333333   
3     ARI    2016     4          42.250000            0.500000   
4     ARI    2016     5          42.444444            0.666667   
5     ARI    2016     6          40.000000            0.400000   
6     ARI    2016     7          38.166667            0.500000   
7     ARI    2016     8          43.166667            0.166667   
8     ARI    2016    10          44.250000            0.250000   
9     ARI    2016    11          44.800000            0.600000   

   punt_distance_roll6  inside_twenty_rate_roll6  
0                  NaN                       NaN  
1            36.000000                  0.600000  
2            41.250000                  0.550000  
3            39.277778                  0.477778  
4            40.02

In [10]:
punts = punts.merge(game_punt_stats[["posteam", "season", "week", "punt_distance_roll6", "inside_twenty_rate_roll6"]],on=["posteam", "season", "week"], how="left")

print(punts.shape)
print(punts[["yardline_100", "punt_distance_roll6", "inside_twenty_rate_roll6", "opponent_start"]].head(10))

(19530, 48)
   yardline_100  punt_distance_roll6  inside_twenty_rate_roll6  opponent_start
0          39.0            47.623611                  0.448611            96.0
1          56.0            44.038889                  0.338889            92.0
2          53.0            45.526984                  0.374603            97.0
3          46.0            45.526984                  0.374603            92.0
4          46.0            48.500000                  0.444444            97.0
5          42.0            43.858333                  0.616667            99.0
6          46.0            45.736111                  0.516667            67.0
7          73.0            44.938889                  0.323611            73.0
8          74.0            44.938889                  0.323611            67.0
9          74.0            44.938889                  0.323611            87.0


In [11]:
punts = punts.dropna(subset=["punt_distance_roll6", "inside_twenty_rate_roll6"]).copy()
print(punts.shape)

(19388, 48)


### Train/Test Split

In [12]:
features = ["yardline_100", "punt_distance_roll6", "inside_twenty_rate_roll6"]
target = "opponent_start"

train = punts[punts["season"] <= 2022]
test = punts[punts["season"] >= 2023]

X_train = train[features]
y_train = train[target]
X_test = test[features]
y_test = test[target]

print(f"Training rows: {len(X_train)}")
print(f"Testing rows: {len(X_test)}")

model = xgb.XGBRegressor(
    n_estimators = 300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42
)

model.fit(X_train, y_train)
print("Model trained")

Training rows: 15099
Testing rows: 4289
Model trained


In [15]:
y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print(f"RMSE: {rmse:.2f} yards")
print(f"MAE: {mae:.2f

importance = model.feature_importances_
for name, score in zip(features, importance):
    print(f"{name}: {score:.3f}")

import os
os.makedirs("../models", exist_ok = True)
model.save_model("../models/punt_outcome_xgb.json")
print("Model saved")

yardline_100: 0.943
punt_distance_roll6: 0.032
inside_twenty_rate_roll6: 0.025
Model saved
